In [40]:
import operator
from google import genai
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import TypedDict, Literal, Annotated
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

In [41]:
load_dotenv()

client = genai.Client()

In [42]:
class tweetState(TypedDict):

    tweet_topic: str
    tweet: str
    tweet_evaluation: Literal['approved', 'needs_improvement']
    tweet_feedback: dict

    iteration: int
    max_iteration: int

    tweet_feedback_history: Annotated[dict[str, dict[str,str]], operator.ior]

In [43]:
def generate_tweet(State: tweetState) -> tweetState:

    tweet_topic = State['tweet_topic']

    prompt = f"""You are a social media copywriter who writes tweets.

            Task: Write a single tweet about the topic provided below.

            Topic: {tweet_topic}

            Guidelines:
            - Keep it under 280 characters.
            - Write about the topic in a natural way.
            - Feel free to use 2-3 hashtags if relevant.
            - You can add an emoji or two if it fits the vibe.
            - Share useful information about the topic.

            Now generate the tweet."""

    response = client.models.generate_content(
        model = "gemini-3.6-flash",
        contents = prompt
    )

    return {
        'tweet': response.text
    }

In [44]:
class evaluate_tweet_structured_output(BaseModel):

    tweet_evaluation: Literal['approved', 'needs_improvement'] = Field(description = "The evaluation of the tweet")
    failed_criterion: str = Field(description = "On which criterion does the tweet failed")
    reason: str = Field(description = "The reason why the tweet failed")

In [45]:
def evaluate_tweet(State: tweetState) -> tweetState:

    tweet = State['tweet']
    iteration = State['iteration']

    prompt = f"""You are a strict social media quality reviewer for X (Twitter) posts. Your job is to evaluate the tweet below and decide if it is ready to publish or needs revision.

            Tweet to evaluate:
            "{tweet}"

            Evaluate strictly against these criteria:

            1. **Character limit** — Must be under 280 characters. Count exactly.
            2. **Hook strength** — First line must grab attention (curiosity, bold claim, relatable pain point, or surprising fact). Generic openers = fail.
            3. **Clarity** — Message must be clear and easy to understand in one read. No confusing phrasing or ambiguity.
            4. **Tone** — Must sound natural and conversational, not robotic, salesy, or overly formal.
            5. **Filler/cliché check** — Reject if it uses generic phrases like "In today's world," "Exciting news," "Game changer," etc.
            6. **Emoji/hashtag usage** — Max 1 emoji, max 2 hashtags, and only if they add real value. Excessive or forced use = fail.
            7. **Engagement factor** — Should end with a question, CTA, or thought-provoking line (unless purely informational content, which is acceptable).
            8. **Originality** — Should not sound like a generic AI-generated summary; must feel human-written.
            9. **Relevance** — Content must stay on-topic and not drift into unrelated points.

            Grading rule: If the tweet fails **even one** of the above criteria, it must be marked as "Needs Improvement" — do not approve tweets with any weak points.

            Output strictly in this format:

            Verdict: Approved / Needs Improvement
            Failed Criteria: [list which numbered criteria failed, or "None"]
            Reason: [1-2 sentence explanation]
            Suggested Fix: [brief actionable suggestion if Needs Improvement, else "N/A"]

            Do not include anything outside this format."""


    response = client.models.generate_content(
        model = "gemini-3.6-flash",
        contents = prompt,
        config = {
            "response_mime_type": "application/json",
            "response_schema": evaluate_tweet_structured_output
        }
    )

    dict = {
        'failed_criterion': response.parsed.failed_criterion,
        'reason': response.parsed.reason
    }

    dict_history = {
        f"iteration_{iteration}": {"failed_criterion": response.parsed.failed_criterion,
                                   "reason": response.parsed.reason}
    }

    return {
        'tweet_evaluation': response.parsed.tweet_evaluation,
        'tweet_feedback': dict,
        'tweet_feedback_history': dict_history
    }

In [46]:
def optimize_tweet(State: tweetState) -> tweetState:

    tweet = State['tweet']
    tweet_feedback = State['tweet_feedback']

    prompt = f"""You are an expert social media copywriter specializing in rewriting and optimizing tweets based on editorial feedback.

            You will be given:
            1. The original tweet
            2. Structured feedback from a strict reviewer (failed criteria and reason)

            Original Tweet:
            "{tweet}"

            Reviewer Feedback:
            - Failed Criteria: {tweet_feedback['failed_criterion']}
            - Reason: {tweet_feedback['reason']}

            Your task: Rewrite the tweet to fully resolve every issue mentioned in the feedback, while preserving the original topic, intent, and core message.

            Strict rules while rewriting:
            1. **Fix only what is flagged** — don't change parts of the tweet that weren't called out as problems, unless necessary to preserve flow after edits.
            2. **Character limit** — Final tweet must be under 280 characters. Count exactly before finalizing.
            3. **Hook** — Ensure the first line grabs attention (curiosity, bold claim, surprising fact, or relatable pain point) if this was flagged.
            4. **Clarity** — Rewrite any confusing or ambiguous phrasing into single-read-clear language.
            5. **Tone** — Keep it conversational and natural — never robotic, salesy, or overly formal.
            6. **Remove clichés/filler** — Eliminate generic phrases like "In today's world," "Exciting news," "Game changer," etc., if flagged.
            7. **Emoji/hashtag limits** — Max 1 emoji, max 2 hashtags, only if they genuinely add value. Remove excess or forced ones.
            8. **Engagement close** — End with a question, CTA, or thought-provoking line, unless the topic is purely informational.
            9. **Originality** — Must read as human-written, not like a generic AI summary.
            10. **Stay on-topic** — Do not drift from the original subject matter while fixing issues.

            Additional constraints:
            - Do NOT introduce new claims, facts, or information not present in the original tweet.
            - Do NOT change the fundamental meaning or stance of the tweet.
            - If a suggested fix conflicts with any rule above (e.g., suggests exceeding character limit), prioritize the rule over the suggestion.

            Output ONLY the revised tweet text — no explanations, no preamble, no quotation marks, no meta-commentary."""

    response = client.models.generate_content(
        model = "gemini-3.5-flash",
        contents = prompt
    )

    iteration = State['iteration'] + 1

    return {
        'tweet': response.text,
        'iteration': iteration
    }

In [47]:
def check_tweet_evaluation(State: tweetState) -> Literal['approved', 'needs_improvement']:

    if State['tweet_evaluation'] == "approved" or State['iteration'] >= State['max_iteration']: 
        return "approved"
    else:
        return "needs_improvement"

In [48]:
graph = StateGraph(tweetState)

graph.add_node('generate_tweet', generate_tweet)
graph.add_node('evaluate_tweet', evaluate_tweet)
graph.add_node('optimize_tweet', optimize_tweet)

graph.add_edge(START, 'generate_tweet')
graph.add_edge('generate_tweet', 'evaluate_tweet')
graph.add_conditional_edges('evaluate_tweet', check_tweet_evaluation, {"approved": END, "needs_improvement": 'optimize_tweet'})
graph.add_edge('optimize_tweet', 'evaluate_tweet')

workflow = graph.compile()

In [53]:
initial_state = {
    'tweet_topic': "Data safety in the Satellite Internet era {in context of Starlink}",
    'iteration': 1,
    'max_iteration': 5
}

final_state = workflow.invoke(initial_state)

final_state

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 52.139339848s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'},